# 99.7b Spatial Discretization

## Contents

**Tools:** CADET-Python; convergence and sharp-front explorer widgets; loading reference data; refinement and plotting helpers.

**Python used:** Imports, lists and dictionaries, keyword arguments, helper calls and widget display.

**Introduces:** Finite volumes (FV), discontinuous Galerkin (DG), WENO reconstruction, polynomial degree, experimental order of convergence (EOC), and balanced axial and particle resolution.

1. FV and DG solutions of concentration profiles
2. Accuracy and runtime in a load-wash-elute example
3. Grid refinement and experimental convergence order
4. Sharp fronts: WENO reconstruction and DG oscillations
5. Competitive Langmuir binding and self-sharpening fronts
6. General rate models: balancing axial and particle discretization

Spatial discretization affects both the accuracy and computational cost of a simulation. In this lesson, we compare FV and DG methods, investigate how they handle smooth profiles and sharp fronts, and explore how spatial refinement influences accuracy and runtime. Finally, we examine how to balance the resolution between the column and the particles in more complex models.

## Spatial Discretization Methods

There are two spatial discretization methods implemented in CADET-Core: **Discontinuous Galerkin (DG)** and **Finite Volumes (FV)**. 

- **Finite Volumes (FV):** Divides the spatial domain into cells and represents the solution using **volume averages**. Higher-order accuracy can be achieved using **reconstruction** schemes such as **Weighted Essentially Non-Oscillatory (WENO)**.
- **Discontinuous Galerkin (DG):** Divides the spatial domain into **elements** and represents the solution using **polynomials of degree $p$** within each element.

The choice of discretization method, spatial resolution and polynomial degree affects both **accuracy** and **runtime** of the simulation. For further details, see [<u>CADET spatial discretization documentation</u>](https://cadet.github.io/master/interface/spatial_discretization_methods.html).

![approximation_FV_vs_DG.png](data/01_spatial_discretization/approximation/approximation_FV_vs_DG.png)

<details>
<summary><b>Click for more details</b></summary>

![approximation_FV_DG_orders.png](data/01_spatial_discretization/approximation/approximation_FV_DG_orders.png)
![approximation_WENO_vs_DG.png](data/01_spatial_discretization/approximation/approximation_WENO_vs_DG.png)

</details>

## How to select spatial discretization?

In the following, we will compare the accuracy, stability, and computational performance of DG and FV.

## <span style="color: red;"> **Important Note:**</span>
Throughout this notebook, the **time discretization error was fixed** so that the spatial discretization error always dominates. That way, we can investigate spatial discretization separately.

Combining the insights of this notebook and the [time discretization notebook](99.7c_time_discretization_lesson.ipynb), **spatial and temporal** discretization parameters should be **selected to achieve similar accuracy**.
Otherwise, the **larger error dominates** and the computational requirements of the other discretizations attempted lower error renders the **simulation inefficient**.

### Example: Lumped Rate Model with Pores

We start with a **load-wash-elute example** with three components and a salt, modeled by a [<u>Lumped Rate Model with Pores</u>](https://cadet.github.io/v5.1.X/modelling/unit_operations/lumped_rate_model_with_pores.html) with [<u>Steric Mass-Action Binding</u>](https://cadet.github.io/v5.1.X/modelling/binding/steric_mass_action.html#steric-mass-action-model).

**Error vs wall clock time:**

The error is computed over all components $k$ and time points $t_i$ as the absolute discrete $L^\infty$ error

$$
e = \max_{i,k} \left|c_k(t_i) - c_{k,\mathrm{ref}}(t_i)\right|,
$$

Plotting that error against the compute time shows which method is computationally superior:
**the superior method reaches a given accuracy fastest**

In [ ]:
from IPython.display import display

import utility.cadet_runtime as cadet_runtime
from utility.convergence_explorer import convergence_explorer

# Only needed to rerun a level; the published results load without CADET.
cadet_runtime.configure()

lrmp_explorer = convergence_explorer('lrmp_sma')
display(lrmp_explorer)

### Key takeaway: DG can be more efficient by 1-2 orders of magnitude

## Convergence order

What drives the efficiency of numerical methods for smooth settings (i.e. solutions without discontinuous or steep concentration profiles), is the theoretical convergence order of the method.
The experimental order of convergence (EOC) is computed by
$$
\mathrm{EOC} = 
\frac{\log\left(\dfrac{e_{n-1}}{e_n}\right)}
{\log\left(\dfrac{N_n}{N_{n-1}}\right)},
$$
where $e_n$ is the maximum outlet error for refinement $n$, and $N_n$ is the axial resolution. Positive EOC indicates decreasing error.\
For FV: $N_n$ is the number of axial cells.\
For DG: $N_n$ is the number of axial elements. At fixed polynomial degree, using $(p+1)N_n$ gives the same EOC.

## Convergence Study Example:

Consider a **single-component Lumped Rate Model without pores (LRM)** with **linear adsorption**. A concentration pulse is injected for $10 \mathrm{s}$, and the resulting outlet concentration profile is simulated.

To investigate spatial convergence, we progressively refine the axial discretization while keeping the physical and solver parameters fixed.


The numerical solutions are compared against the CASEMA reference solution using the discrete absolute $L^\infty$ error:

$$
e_n = \max_i \left|c_n(t_i)-c_{\mathrm{CASEMA}}(t_i)\right|,
$$

evaluated at the reference output times without normalization. The experimental order of convergence (EOC) is calculated between consecutive spatial refinements.

Observe:
- How **quickly** does the error decrease with increasing spatial resolution?
- How do the experimental convergence orders **compare** between FV and DG?
- Does a higher polynomial degree always lead to better **computational efficiency**?

In [ ]:
from pathlib import Path

from utility.spatial_convergence import (
    load_reference,
    run_refinement_study,
    plot_refinement_study,
    print_max_slopes,
)

# Configure the Cadet runtime and set up the data directory for the reference data
cadet_runtime.configure()
data_directory = Path('data') / '01_spatial_discretization'
reference_file = (data_directory / 'reference'
                  / 'LRM_dynLin_1comp_benchmark1.h5')

# One run per method and grid is written here while the study is
# refined; the published figure needs none of them afterwards.
refinement_directory = (Path('generated') / '01_spatial_discretization'
                        / 'refinement')
refinement_directory.mkdir(parents=True, exist_ok=True)

# Define the axial cell counts and the numerical methods to be used in the study
axial_cell_counts = [4, 8, 16, 32, 64, 128, 256, 512]
methods = {'FV': 0, 'DG1': 1, 'DG2': 2, 'DG3': 3}

# Load the reference solution and run the spatial refinement study
reference = load_reference(reference_file)
results = run_refinement_study(reference, axial_cell_counts, methods,
                               refinement_directory)
fig = plot_refinement_study(reference, axial_cell_counts, results)

In [ ]:
# Print the maximum slopes for the convergence study
print_max_slopes(axial_cell_counts, results)

From the EOC formula, the **estimated error decrease** $\Delta e_n \coloneq e_n - e_{n-1}$ between consecutive refinement levels for a given convergence order $p$ is
$$
\Delta e_n \approx e_{n-1} \cdot \left( \frac{1}{\left(\Delta N_n\right)^p} - 1 \right),
$$
where $\Delta N_n = \frac{N_n}{N_{n-1}}$ denotes the refinement factor.

**This formula can be used to estimate the required discretization for a desired accuracy**.
It has to be treated with **caution** though, because the **convergence order** is an asymptotic property of the transport operators alone and **may change** depending on the error metrics, e.g. if hte error is computed for the chromatogram or bulk solution.

## Why does CADET have FV then? -> Spatial stabilization !

- DG is a **linear high order** method and thus generally allows **new extrema**
- Controlling these extrema requires a nonlinear mechanism like the **WENO FV reconstruction**
- DG is only advantageous if the **solution** is **smooth** enough, which to our experience is **usually** the case

### But when does DG show this bad behaviour ?
**Sharp concentration fronts** are challenging for numerical methods. Low-order schemes often introduce **numerical diffusion**, smoothing sharp fronts, while higher-order schemes may produce **spurious oscillations** near steep gradients.  


<details>
<summary><b>Click for details on WENO</b></summary>
The **FV Weighted Essentially Non-Oscillatory (WENO)** scheme addresses the high-order loworder trade-off by combining several reconstruction stencils using smoothness-dependent weights. This allows high-order accuracy in smooth regions while reducing oscillations near sharp gradients. 

CADET provides three WENO reconstruction settings for FV method,controlled by `WENO_ORDER`:

| `WENO_ORDER` | Stencil size | Order of accuracy |Typical behavior |
|---|---|---|---|
| **1** | 1 cell (first-order upwind) | 1st |  Robust, but introduces substantial numerical diffusion |
| **2** | 2-cell stencils | 3rd (2nd order at boundaries) | Sharper fronts, reduced numerical diffusion |
| **3** | 3-cell stencils | 5th (2nd order at boundaries) | Higher accuracy in smooth regions, potentially sharper fronts |


Increasing the reconstruction order can improve the resolution of sharp concentration fronts, but may also increase computational cost. Further reading: [<u>Jiang & Shu (1996), Efficient Implementation of Weighted ENO Schemes.</u>](https://doi.org/10.1006/jcph.1996.0130)

Let's investigate how these schemes behave when transporting sharp concentration fronts through a column.

### WENO Example: Sharp Fronts and Pulses

Consider a $0.1 \mathrm{m}$ column without adsorption or particles, initially filled with pure solvent. A solute is introduced at the inlet and transported through the column by convection.

**What happens when axial dispersion is very small?**

With little physical smoothing, sharp concentration changes remain steep as they travel through the column. The numerical method must therefore capture these fronts without introducing excessive smearing or artificial oscillations.

We investigate two scenarios:

**1. Step input:** A constant concentration of $1,\mathrm{mol/m^3}$ is introduced, creating a single advancing front.

**2. Pulse input:** The solute is injected for $60,\mathrm{s}$, followed by pure solvent, creating both leading and trailing fronts.

Compare WENO1, WENO2, and WENO3 using the same axial degrees of freedom and dispersion coefficient:

- Run the simulations and compare the concentration profiles.
- Observe how sharply each scheme captures the fronts.
- Look for numerical diffusion and possible undershoots or overshoots.
- Change the spatial resolution and dispersion coefficient. How does the behavior change?

**Reference:** The dashed black curve is what the methods are judged against, and it follows the dispersion slider.

At $D = 0$ there is nothing to solve: the column only delays the inlet signal by the residence time, so the reference is written down exactly,

$$
\tau = \frac{L}{u} \approx 88.1\,\mathrm{s},
\qquad
c_{\mathrm{out}}(t) = c_{\mathrm{in}}(t-\tau).
$$

At any $D > 0$ the outlet is a smeared front with no usable closed form, so the reference is **the same model on a much finer grid**: finite volumes with WENO3, resolving the front with about ten cells. That scheme is monotone, so the curve everything is measured against can never overshoot on its own account. The legend names the grid it used, and the explorer warns when the front is too thin for even that grid to resolve, which happens at the sharpest settings.

**Numerical behavior:** The concentration should remain between $0$ and $1,\mathrm{mol/m^3}$. Values outside this range indicate numerical **undershoot** or **overshoot**, while excessive smoothing may indicate **numerical diffusion**.

In [ ]:
import importlib
from IPython.display import display
from utility.sharp_front_comparison import sharp_gradient_explorer

# Set up the WENO methods
weno_methods = [('FV', 1), ('FV', 2), ('FV', 3)]

# Run two studies for the WENO methods
weno_explorer = sharp_gradient_explorer(
    mode='both',
    output_directory=(Path('generated') / '01_spatial_discretization'
                      / 'rerun'),
    methods=weno_methods,
)
display(weno_explorer)

### Discontinous Galerkin 

Unlike the finite volume (FV) method, which represents the solution using cell averages and reconstructs values at cell interfaces, the **Discontinuous Galerkin (DG)** method approximates the solution using polynomials within each element. The polynomial degree $p$ controls the approximation order.

Higher polynomial degrees can provide greater accuracy for smooth solutions. However, near sharp concentration fronts or discontinuities, polynomial approximations may produce **spurious oscillations**. This occurs because a finite-degree polynomial cannot represent a discontinuity within an element exactly, and its approximation is not necessarily bounded by the minimum and maximum values of the solution. This behavior is related to the **Gibbs phenomenon**.

Let's investigate this behavior by comparing **DG** with **FV (WENO3)** for the same sharp-front and pulse examples.

### DG Example: Sharp Fronts and Pulses

Let's revisit the previous examples using the same column and inlet conditions.

Compare **DG**, **FV (WENO3)** and the reference for the dispersion you selected:
- Look for numerical overshoots and undershoots. Do you observe any **oscillations**? 
- How do the oscillations behave with respect to changing **dispersion coefficient**, **polynomial degree** and **resolution**?
- Does increasing the polynomial degree always improve the solution? 
- How does **DG** compare with **FV (WENO3)** in terms of accuracy, oscillations, and computational cost?

In [ ]:
# Set up the DG methods for the comparison
dg_methods = [('FV', 3),
              ('DG', 1), ('DG', 2), ('DG', 3), ('DG', 4), ('DG', 5)]

# Run the DG methods comparison study
dg_explorer = sharp_gradient_explorer(
    mode='both',
    output_directory=(Path('generated') / '01_spatial_discretization'
                      / 'rerun'),
    methods=dg_methods,
)
display(dg_explorer)

### Key-takeaway: With enough dispersion, there are no strong gradients
Dispersion comes from
- axial dispersion coefficient
- film diffusion kinetics
- pore diffusion kinetics
- binding kinetics

### Do strong gradients occur in chromatographic separations?

### Example: Competitive Langmuir with steep concentration fronts

The same comparison for a **two-component [<u>Langmuir</u>](https://cadet.github.io/v5.1.X/modelling/binding/multi_component_langmuir.html) setting** in a [<u>Lumped Rate Model</u>](https://cadet.github.io/v5.1.X/modelling/unit_operations/lumped_rate_model_without_pores.html), with little axial dispersion. The solution develops **self-sharpening concentration fronts**, and the convergence plot looks nothing like the smooth case from before:

In [ ]:
from IPython.display import display

import utility.cadet_runtime as cadet_runtime
from utility.convergence_explorer import convergence_explorer

cadet_runtime.configure()

langmuir_explorer = convergence_explorer('lrm_langmuir')
display(langmuir_explorer)

### Key takeaway: DG struggles with strong gradients, FV can be more efficient here

## Spatial discretization of multiple domains

### Example: General Rate Model for the same load-wash-elute case

The following example is again a load-wash-elute simulation with [<u>Steric Mass-Action Binding</u>](https://cadet.github.io/v5.1.X/modelling/binding/steric_mass_action.html#steric-mass-action-model).
**The [<u>General Rate Model</u>](https://cadet.github.io/v5.1.X/modelling/unit_operations/general_rate_model.html) spatially resolves the particle radius**, which also needs to be discretized.
CADET-Core discretizes the bulk and particle coordinates independently.
That is, we can combine different DG polynomial degrees for bulk and particle, or even combine FV and DG.
While this increases **approximation flexibility **and allows **granular refinement** of the spatial grid, it further **complicates** the **choice** of the **best** possible **discretization**.

In the following performance-accuracy benchmark, try the following:
- Fix the bulk discretization to FV with 16 cells, choose FV for the particles. Hit "Clear history" and refine the particle by moving the number of cells. How does the error evolve?
- Fix the particle discretization to FV with 32 cells, choose DGP3 for the bulk. Hit "Clear history" and refine the bulk by moving the number of cells. How does the error evolve?
- Try to find the optimal (i.e. computationally fastest) discretization that achieves an approximation $L^\infty$ error $<1\%$
- What discretization strategy seems to work well? Hit "add all points" to see the pareto front.

In [ ]:
from pathlib import Path

from IPython.display import display

import utility.cadet_runtime as cadet_runtime
from utility.lwe_discretization_explorer import lwe_discretization_explorer

# Only needed to rerun a setting; the stored results load without CADET.
cadet_runtime.configure()

discretization_explorer = lwe_discretization_explorer(
    store=(Path('data') / '01_spatial_discretization'
           / 'lwe_discretization'),
    output_directory=(Path('generated') / '01_spatial_discretization'
                      / 'rerun'),
)
display(discretization_explorer)

### Key-takeaway: Axial and particle discretization must be in balance

With one domain there was a single ladder to climb. With two there is a **Pareto front**, and the optimum is genuinely hard to guess.

- **Five per cent of the grid is on the pareto front**
- **FV and DG1** are **not on the front** below an $L^\infty$ error of $3 \%$ (this is again a smooth setting)
- Particle and bulk discretization must be in balance and the optimal choice might be to **combine different methods**

| bulk / par method | ax. cells | par. cells | DoF per comp. | $L^\infty$ error | time |
|---|---|---|---|---|---|
| **FV / FV** | 1 | 1 | 2 | 363 % | **0.01 s** |
| FV / FV | 1 | 2 | 3 | 261 % | 0.01 s |
| FV / FV | 2 | 1 | 4 | 165 % | 0.02 s |
| FV / FV | 2 | 2 | 6 | 95.3 % | 0.02 s |
| FV / FV | 2 | 4 | 10 | 56.8 % | 0.05 s |
| FV / DG1 | 2 | 1 | 6 | 45.2 % | 0.05 s |
| FV / FV | 4 | 2 | 12 | 28.7 % | 0.07 s |
| DG2 / DG1 | 1 | 1 | 9 | 26.5 % | 0.11 s |
| FV / FV | 4 | 4 | 20 | 25.8 % | 0.13 s |
| FV / FV | 8 | 2 | 24 | 14.7 % | 0.20 s |
| DG2 / FV | 2 | 2 | 18 | 10.2 % | 0.21 s |
| DG2 / DG1 | 2 | 1 | 18 | 6.17 % | 0.27 s |
| DG2 / DG1 | 3 | 1 | 27 | 4.95 % | 0.47 s |
| DG3 / DG1 | 2 | 1 | 24 | 4.92 % | 0.48 s |
| DG2 / DG2 | 2 | 1 | 24 | 3.29 % | 0.51 s |
| DG2 / DG1 | 2 | 2 | 30 | 3.22 % | 0.67 s |
| DG2 / DG3 | 2 | 1 | 30 | 2.16 % | 0.77 s |
| DG3 / DG2 | 2 | 1 | 32 | 1.52 % | 0.80 s |
| DG4 / DG2 | 2 | 1 | 40 | 1.38 % | 1.12 s |
| DG3 / DG3 | 2 | 1 | 40 | 0.958 % | 1.15 s |
| DG2 / DG3 | 3 | 1 | 45 | 0.695 % | 1.39 s |
| DG4 / DG3 | 2 | 1 | 50 | 0.268 % | 1.59 s |
| DG3 / DG3 | 3 | 1 | 60 | 0.207 % | 1.77 s |
| DG5 / DG4 | 2 | 1 | 72 | 0.185 % | 2.83 s |
| DG3 / DG4 | 3 | 1 | 72 | 0.125 % | 2.84 s |
| DG3 / DG5 | 3 | 1 | 84 | 0.0252 % | 3.30 s |
| DG4 / DG5 | 3 | 1 | 105 | 0.0099 % | 4.36 s |
| DG4 / DG3 | 3 | 3 | 195 | 0.00904 % | 10.8 s |
| DG3 / DG3 | 4 | 3 | 208 | 0.00836 % | 11.5 s |
| DG4 / DG5 | 3 | 2 | 195 | 0.00743 % | 12.3 s |
| DG5 / DG3 | 3 | 3 | 234 | 0.00664 % | 13.0 s |
| DG5 / DG5 | 3 | 2 | 234 | 0.00341 % | 15.8 s |
| DG4 / DG4 | 4 | 3 | 320 | 0.00319 % | 18.9 s |
| DG4 / DG5 | 4 | 2 | 260 | 0.00315 % | 21.0 s |
| DG5 / DG5 | 3 | 3 | 342 | 0.00299 % | 22.0 s |
| DG4 / DG5 | 4 | 3 | 380 | 0.00227 % | 24.0 s |
| DG5 / DG5 | 4 | 3 | 456 | 0.000951 % | 29.4 s |
| **DG4 / DG5** | 4 | 4 | 500 | **0.000399 %** | **36.4 s** |

## Back to oscillations - negative approximation errors

- So far, approximation quality was measured in terms of $L^\infty, L^1, L^2$ error norms.
- an **acceptable $L^\infty, L^1, L^2$ error** can still be a problem when **concentrations become negative**.
- Negative concentrations have **no physical meaning**.
- Even worse, some binding or reaction models might **crash the simulation** when negative values occur.

### Example 1: a binding model that cannot survive an undershoot

This is the setting from
[<u>CADET forum thread 1283</u>](https://forum.cadet-web.de/t/error-with-dg-discretization-and-empm-binding-model/1283),
where a DG simulation fails and the same model runs under FV.

A lumped rate model without pores, two components, loaded for 200 s and then
washed. Binding is the
[<u>Extended Mobile Phase Modulator</u>](https://cadet.github.io/v5.1.X/modelling/binding/extended_mobile_phase_modulator.html),
whose isotherm contains factor depending on the ion exchange characteristic $\beta$:

$$c_{\mathrm{salt}}^{\beta}$$

Task:
- Run DG and FV simulations for different values of $\beta$

In [ ]:
from pathlib import Path

from IPython.display import display

import utility.cadet_runtime as cadet_runtime
from utility.negative_concentration_explorer import (
    forum1283_case, negative_concentration_explorer)

cadet_runtime.configure()

# Every setting here is simulated live: this model is small and a run takes
# well under a second. "Fill this ladder" runs every resolution of the
# selected method, which is what draws a full curve in the upper panels.
empm_explorer = negative_concentration_explorer(
    forum1283_case(),
    output_directory=(Path('generated') / '01_spatial_discretization'
                      / 'negatives' / 'forum1283'),
)
display(empm_explorer)

### Key takeaway: some binding or reaction models require positivity preserving spatial discretization

## Summary

- DG has arbitrary (selecteable) theoretical convergence order and can thus be significantly more efficient for smooth concentration profiles
- FV can be more efficient and stable for sharp concentration fronts
- Some models require a stable spatial discretization such as WENO FV
- Bulk and particle resolution must be belanced for optimal computational performance

**Next lesson:** [**99.7c Time Discretization**](99.7c_time_discretization_lesson.ipynb) explores implicit time integration, the influence of error tolerances on accuracy and runtime, and the structure of the Jacobian matrix.

## References
- **Original WENO FV CADET code**: [Eric von Lieres and Joel Andersson. A fast and accurate solver for the general rate model of column liquid chromatography. 2010.](https://doi.org/10.1016/j.compchemeng.2010.03.008)
- **First DGSEM implementation for chromatography**: [Kristian Meyer, Samuel Leweke, and Eric von Lieres, Jakob K. Huusom, Jens Abildskov. ChromaTech: A discontinuous Galerkin spectral element simulator for preparative liquid chromatography. 2020.](https://doi.org/10.1016/j.compchemeng.2026.109928)
- **CADET DGSEM implementation and first investigation on oscillations**: [Jan Michael Breuer, Samuel Leweke, Johannes Schmölder, Gregor Gassner, and Eric von Lieres. Spatial discontinuous galerkin spectral element method for a family of chromatography models in CADET. 2023.](https://doi.org/10.1016/j.compchemeng.2023.108340)